# Populate csv file with txt transcripts
## **need**
- csv with transcripts column that needs to be populated
- transcript column name
- txt title id structure (how to correlate txt title to the column name)
- folder location of txt files

## Inputs
- **csv_path**: path to the CSV file to update
- **transcript_col**: column name to populate (e.g., `transcript`)
- **txt_folder_path**: folder containing `.txt` files
- **id rule**: how the text filename maps to the CSV id (example below uses filename stem)

## Expected CSV columns
- An identifier column to match files (example: `id`)
- The transcript column to fill

## Notes
- This example reads each `.txt` file and joins on `id` == filename stem.
- Adjust `id_col` and `file_id_from_name()` to match your naming scheme.

In [ ]:
import ast
from pathlib import Path
import pandas as pd
import csv

# edit these fields accordingly
dataset_name = "isleevy"
csv_path = r"../data/isleevy/original_data/isleevy-220329.csv"
txt_folder_path = r"../data/isleevy/isleevy-txts/"

# these should remain the same
id_col = "SSID"
transcript_col = "Transcript"

In [169]:

def file_ids_from_name(path: Path) -> list[str]:
    stem = path.stem.strip()
    ids = [stem]

    parts = stem.split("-")
    if len(parts) >= 2:
        ids.insert(0, parts[-2].strip())

    return list(dict.fromkeys([i for i in ids if i]))

def read_transcript(path: Path) -> str:
    raw = path.read_text(encoding="utf-8-sig", errors="ignore")
    if not raw:
        return ""

    # Unwrap list-literal files if needed.
    raw = raw.strip()
    if raw.startswith("[") and raw.endswith("]"):
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                raw = "\n".join(str(item) for item in parsed)
        except (ValueError, SyntaxError):
            pass

    return raw

In [170]:
# Detect source delimiter to avoid accidental single-column parses.
with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
    sample = f.read(65536)
try:
    detected_delimiter = csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t", "|"]).delimiter
except csv.Error:
    detected_delimiter = ","

csv_file = pd.read_csv(csv_path, sep=detected_delimiter)

txt_dir = Path(txt_folder_path)
txt_map = {}

for p in txt_dir.glob("*.txt"):
    txt = read_transcript(p)
    for file_id in file_ids_from_name(p):
        txt_map.setdefault(file_id, txt)

id_series = csv_file[id_col].astype(str).str.strip()
mapped = id_series.map(txt_map)

fallback_ids = id_series.str.replace(r"\.txt$", "", regex=True)
mapped = mapped.fillna(fallback_ids.map(txt_map))

# Keep raw transcript content, but encode line breaks to avoid row-splitting in CSV consumers.
safe_transcripts = (
    mapped.fillna("")
    .astype(str)
    .str.replace("\x00", "", regex=False)
    .str.replace("\r\n", "\n", regex=False)
    .str.replace("\r", "\n", regex=False)
    .str.replace(r"\\+n", "\n", regex=True)   # fold any number of backslashes before n into a real newline
    .str.replace("\n", r"\\n", regex=False)   # then escape once uniformly
)
csv_file[transcript_col] = safe_transcripts

# Normalize empty non-transcript fields so trailing empty-last-column values don't confuse brittle importers.
for col in csv_file.columns:
    if col != transcript_col:
        csv_file[col] = csv_file[col].replace(r"^\s*$", pd.NA, regex=True).fillna("nan")

output_path = Path(csv_path).with_name(f"{dataset_name} metadata.csv")
with output_path.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(
        f,
        delimiter=detected_delimiter,
        quoting=csv.QUOTE_ALL,
        quotechar='"',
        doublequote=True,
        lineterminator="\n",
    )
    writer.writerow(csv_file.columns.tolist())
    writer.writerows(csv_file.itertuples(index=False, name=None))

print(f"Read delimiter: {repr(detected_delimiter)}")
print(f"Wrote: {output_path}")
print(f"Rows written (excluding header): {len(csv_file)}")

Read delimiter: ','
Wrote: ../data/isleevy/original_data/isleevy metadata.csv
Rows written (excluding header): 27


In [171]:
csv_file

,SSID,Filename,Tags,Title,Title Confidence,Creator,Creator Confidence,Volume,Volume Confidence,Issue,...,Period Confidence,Location,Location Confidence,Named Entities,Named Entities Confidence,Keywords,Keywords Confidence,Resource Type,Resource Type Confidence,Transcript
0,42035475,isleevy-009.jpg,AI Generated,Marriage Invitation for Mary E. Kirkland and I...,high,Mr. and Mrs. Levi Kirkland,high,nan,nan,nan,...,low,"Westville, South Carolina",high,Mary E. Kirkland | Isaac S. Leevy Jr. | Mr. an...,high,marriage | wedding | invitation | South Caroli...,high,Images,nan,=== Page 1 ===\\n7198.\\n[23 June 1909]\\n\\nM...
1,42035482,isleevy-020.jpg,AI Generated,Memorial Obsequies Program for Isaac Samuel Leevy,high,Unknown,medium,nan,nan,nan,...,high,"Columbia, South Carolina, United States",high,Isaac Samuel Leevy | Allen University | Allen ...,medium,Isaac Samuel Leevy | memorial | funeral progra...,high,Images,nan,=== Page 1 ===\\n[NO TEXT DETECTED]\\n\\n=== P...
2,42035496,isleevy-037.jpg,AI Generated,Leevy's Standard Furniture Company Rental Agre...,low,Leevy's Standard Furniture Company | I. S. Leevy,high,nan,nan,nan,...,medium,"Columbia, South Carolina",high,Leevy's Standard Furniture Company | I. S. Lee...,high,furniture rental | merchant tailor | business ...,high,Images,nan,=== Page 1 ===\\n7198\\n\\n[N. W.]\\n\\nI. S. ...
3,42035494,isleevy-030.jpg,AI Generated,"Leevy's Funeral Home, Inc. promotional hand fan",medium,"Leevy's Funeral Home, Inc.",high,nan,nan,nan,...,medium,"Columbia, South Carolina, United States",high,"Leevy's Funeral Home, Inc. | I. S. Leevy | Col...",medium,funeral home | advertising | African American ...,high,Images,nan,"=== Page 1 ===\\n7/98.\\n\\n[c. 1970]\\n\\n""A ..."
4,42035464,isleevy-003.jpg,AI Generated,"Postcards sent to Mary Kirkland, 1906–1908",medium,Unknown | J. S. Leary | Johns Filamore Price,low,nan,nan,nan,...,high,"South Carolina, United States",high,Mary Kirkland | J. S. Leary | Johns Filamore P...,high,postcard | Mary Kirkland | South Carolina | We...,high,Images,nan,=== Page 1 ===\\n[NO TEXT DETECTED]\\n\\n=== P...
5,42035476,isleevy-013.jpg,AI Generated,Leevy's Funeral Home Undertaking and Embalming...,high,Leevy's Funeral Home,high,nan,nan,nan,...,high,"Columbia, South Carolina",high,"Leevy's Funeral Home | I. S. Leevy | Columbia,...",high,funeral home | undertaking | embalming | adver...,high,Images,nan,"=== Page 1 ===\\n7198.\\nc.1940\\n\\n""Leevy Le..."
6,42035488,isleevy-028.jpg,AI Generated,"The Columbia Record, Columbia, South Carolina,...",high,"Montgomery, John A. | Jenkins, H. Harrison",high,nan,nan,nan,...,high,"Columbia, South Carolina, United States",high,Isaac Samuel Leevy | State Fair Association | ...,high,Isaac Samuel Leevy | public service | educatio...,high,Images,nan,=== Page 1 ===\\n7198\\n\\nTHE COLUMBIA RECORD...
7,42035484,isleevy-026.jpg,AI Generated,"I. S. Leevy, Columbia Business Man, Gets Servi...",high,Unknown,high,nan,nan,nan,...,high,"Columbia, South Carolina",high,I. S. Leevy | Standard Oil Company of New Jers...,medium,I. S. Leevy | Columbia | Standard Oil Company ...,high,Images,nan,=== Page 1 ===\\n7199 [N. W.]\\n\\nI. S. Leevy...
8,42035516,isleevy-132.jpg,AI Generated,Compositions: History Notes by I. S. Leevy,medium,I. S. Leevy,high,nan,nan,nan,...,medium,United States,high,I. S. Leevy | E. Pool | Dr. Gordon | U.S. Depa...,medium,history | lecture notes | Grecian history | at...,high,Images,nan,"=== Page 1 ===\\nPLeevy, Isaac Samuel (1877-19..."
9,42035474,isleevy-012.jpg,AI Generated,Christmas and New Year Greeting Card from Good...,high,Goodwin Lumber & Supply Co.,high,nan,nan,nan,...,high,"Columbia, South Carolina",high,Goodwin Lumber & Supply Co. | 1527 Lyon St. | ...,medium,Christmas | New Year | business appreciation |...,high,Images,nan,=== Page 1 ===\\nThe foundation of all busines...


In [173]:
import importlib.util

# Load helper module
spec = importlib.util.spec_from_file_location("sh", "01-seeklight-training-data-helper.py")
sh = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sh)

# Generate both training data formats from processed split data
descriptions_path, transcripts_path = sh.format_training_data(
    csv_input_file_path=str(output_path),
    dataset_name=dataset_name,
    output_path=None,  # Will use output_path parent / training
)


Wrote descriptions_included: ../data/isleevy/training/isl_descriptions_included.csv
Wrote transcripts: ../data/isleevy/training/isl_transcripts.csv
Rows: 27
